<a href="https://colab.research.google.com/github/laoneya31/IAT360-Group-18/blob/main/notebooks/Daksh_Setup_Baseline_BatchSize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><h1><b>Fruit Freshness Detection Using YOLO</b></h1></center>
<center>IAT 360 Group 18: training environment, baseline and batch size (Daksh)</center>

Grocery stores lose money when rotting produce goes unnoticed in display bins and spreads spoilage to nearby fruit. This
project fine-tunes a pretrained YOLOv8 model to detect every apple, banana and orange in a cluttered scene and flag the
rotten ones. It is a **supervised object detection** problem, and the model could run on a camera above a produce display
or a back-room sorting table.

This notebook covers my part of the project:

| Task | Section |
|---|---|
| Training environment: YOLO repo cloned, Colab GPU, shared training script | 1 |
| Quick test training on a small subset, to catch format errors early | 2 to 4 |
| Train and save the baseline, and share its results | 5 |
| Tune batch size at 3 values and compare with the baseline | 6 |
| Export YOLO plots and confusion matrices into one folder | 7 |
| Shortcomings and challenges notes for the report | 8 |

**Real data or practice data.** Sections 5 to 7 use the group dataset as soon as Philip's `fruit_freshness.zip` is on
Drive. Until then they run a **rehearsal** on the small practice set from Section 3, with tiny settings, only to prove the
code works. Rehearsal numbers are not results and must not go in the report.

# 1. Environment

Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**. The shared Drive folder must be in your My Drive: right-click `IAT360-Group 18` in *Shared with me* and choose **Organize > Add shortcut**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone our repo and the official YOLO repo. YOLO is pinned to one release so all four members train with identical code.

In [ ]:
import os, sys
REPO = "/content/IAT360-Group-18"
YOLO_TAG = "v8.4.204"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/laoneya31/IAT360-Group-18 {REPO}
# The folder must not be called "ultralytics": /content is on sys.path, so it would hide the real package.
if not os.path.exists("/content/yolo_repo"):
    !git -c advice.detachedHead=false clone -q --depth 1 --branch {YOLO_TAG} https://github.com/ultralytics/ultralytics /content/yolo_repo
%pip install -q /content/yolo_repo kagglehub
%cd {REPO}

Importing required libraries, and checking the software and hardware.

In [ ]:
import glob, hashlib, random, shutil
import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

import ultralytics
ultralytics.checks()
from ultralytics import YOLO

sys.path.insert(0, f"{REPO}/src")
import fruit_yolo as fy

fy.check_config()
print("Team Drive folder:", fy.find_team_drive())
print("Baseline settings:", fy.BASELINE)

# 2. Dataset Description

The group dataset has six classes. They combine the fruit type with its freshness, following our
[labelling rules](https://github.com/laoneya31/IAT360-Group-18/blob/main/docs/LABELLING_RULES.md).

In [ ]:
class_labels = dict(enumerate(fy.CLASS_NAMES))
class_labels

**Labelling format.** Each image has one `.txt` file with one row per object:

    class_id x_center y_center width height

All four box values are normalised to between 0 and 1, relative to the image size.

**Folder hierarchy.** YOLO expects images and labels in matching folders, and this layout is already in our repo:

    datasets/fruit_freshness/
      images/train   images/val   images/test
      labels/train   labels/val   labels/test

**Where the data comes from.** Public datasets are used for training and validation. Our own 120 photos, 30 per member,
are held out as the test set. Sources are listed in
[DATA_SOURCES.md](https://github.com/laoneya31/IAT360-Group-18/blob/main/docs/DATA_SOURCES.md).

**Practice set for the quick test.** To test the pipeline before the group dataset is ready, I build a small multi-fruit
set in the same format from the Kaggle
[Fruits fresh and rotten](https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification) images. It
is regenerated on each run and is never committed to GitHub.

# 3. Data Preprocessing

## Collecting the source images

In [ ]:
kaggle_root = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
source_root = f"{kaggle_root}/dataset/train"
folders = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]
sources = {i: sorted(glob.glob(f"{source_root}/{name}/*")) for i, name in enumerate(folders)}
for i, name in enumerate(folders):
    print(f"{class_labels[i]:<15} <- {name:<14} {len(sources[i])} photos")

This download holds a second copy of everything in a nested `dataset/dataset/` folder. Checking whether it is a true duplicate, because a recursive search would pull the same photos into both train and test.

In [ ]:
def md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

sample = random.Random(0).sample(sources[0], 200)
nested = [p.replace("/dataset/train/", "/dataset/dataset/train/") for p in sample]
same = sum(os.path.exists(n) and md5(n) == md5(p) for p, n in zip(sample, nested))
print(f"{same} of {len(sample)} sampled files are byte-identical in the nested copy")

## Creating images and text files for labels

Each single-fruit photo has a plain background, so the fruit can be cut out and pasted on a new background. That gives a
multi-fruit scene and an exact bounding box, which is written to a `.txt` file in YOLO format. Source photos are divided
between train, val and test first, so the same photo cannot appear in two splits.

In [ ]:
SMOKE = "/content/datasets/smoke"
SMOKE_CONFIG = "/content/smoke.yaml"
rng = random.Random(0)
subset = {cid: rng.sample(files, 60) for cid, files in sources.items()}

if not os.path.exists(SMOKE):
    print("boxes written:", fy.build_composites(subset, SMOKE, {"train": 120, "val": 30, "test": 30}, seed=0))
fy.write_config(SMOKE, SMOKE_CONFIG)

One label file, to confirm the format matches the description above.

In [ ]:
first_label = sorted(glob.glob(f"{SMOKE}/labels/train/*.txt"))[0]
print(open(first_label).read())

## Visualising the dataset

`draw_yolo_bbox` converts the normalised values back to pixels and draws each box with its label, which confirms the labels survived the conversion.

In [ ]:
def draw_yolo_bbox(image_path, label_path, ax):
    image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    for class_id, x1, y1, x2, y2 in fy.yolo_boxes(label_path, width, height):
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(image, class_labels[class_id], (int(x1), max(14, int(y1) - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    ax.imshow(image)
    ax.axis("off")

fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, path in zip(axes, sorted(glob.glob(f"{SMOKE}/images/train/*.jpg"))[:4]):
    draw_yolo_bbox(path, path.replace("/images/", "/labels/").replace(".jpg", ".txt"), ax)
plt.tight_layout()
plt.show()

## Checking Missing Files

Checking that every image has a label file and every label file has an image.

In [ ]:
for split in ("train", "val", "test"):
    image_files = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{SMOKE}/images/{split}/*.jpg")}
    label_files = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{SMOKE}/labels/{split}/*.txt")}
    print(split, len(image_files), " = ", len(label_files))
    print(f"  Extra images (without corresponding labels): {image_files - label_files}")
    print(f"  Extra labels (without corresponding images): {label_files - image_files}")

## Checking label format

`validate_dataset` goes further. It also catches the errors that do not crash YOLO but silently damage training: class ids
out of range, pixel instead of normalised coordinates, polygon exports, and the same image in two splits. Every member can
run it on any dataset folder before training.

In [ ]:
report = fy.validate_dataset(SMOKE)
fy.assert_valid(report)

A check is only useful if it fails when it should. Here I break a copy of the practice set on purpose, once for each kind of mistake we could plausibly make.

In [ ]:
BROKEN = "/content/datasets/broken"
shutil.rmtree(BROKEN, ignore_errors=True)
shutil.copytree(SMOKE, BROKEN)
labels = sorted(glob.glob(f"{BROKEN}/labels/train/*.txt"))
images = sorted(glob.glob(f"{BROKEN}/images/train/*.jpg"))

open(labels[0], "w").write("6 0.5 0.5 0.2 0.2\n")               # class id that does not exist
open(labels[1], "w").write("0 320 240 50 50\n")                 # pixels, not normalised
open(labels[2], "w").write("0 0.1 0.1 0.2 0.2 0.3 0.3\n")       # polygon export
os.remove(labels[3])                                             # label lost
os.remove(images[4])                                             # image lost
shutil.copy(images[5], f"{BROKEN}/images/val/leaked.jpg")        # train image copied into val
shutil.copy(labels[5], f"{BROKEN}/labels/val/leaked.txt")

broken = fy.validate_dataset(BROKEN, near_duplicates=False)
assert len(broken["errors"]) == 6, broken["errors"]
print("\nAll 6 planted mistakes were caught.")

# 4. Custom object detection using YOLO

## Config file

The YAML file tells YOLO where the images are, how many classes there are, and their names. Ours is committed at the top of
the repo. It has no `path:` line on purpose, so YOLO reads the paths relative to the file and it works wherever the repo is
cloned.

In [ ]:
print(open(fy.CONFIG).read())

## Quick test training

Three short epochs before anyone spends GPU hours: if this runs, the folder layout, labels, config and training code all
work together. The smaller `yolov8n` model and a low image size are used here only to keep this test fast.

In [ ]:
smoke_weights = fy.train_run("smoke_test", "Daksh", data=SMOKE_CONFIG, rehearsal=True)
smoke = fy.evaluate("smoke_test", smoke_weights, data=SMOKE_CONFIG, rehearsal=True)
print({k: round(float(smoke[k]), 3) for k in fy.METRIC_COLUMNS})

# 5. Baseline model

## Training YOLO

Choosing the data. This uses the group dataset if it is ready, and the practice set if not.

In [ ]:
REAL = fy.fetch_dataset()
if REAL:
    fy.require_gpu()
    fy.assert_valid(fy.validate_dataset())
    DATA, ROOT, REHEARSAL = fy.CONFIG, fy.DATASET, False
else:
    DATA, ROOT, REHEARSAL = SMOKE_CONFIG, SMOKE, True
    print("\n*** REHEARSAL: group dataset not ready. Numbers below only prove the code runs. Do not report them. ***")
print("Results go to:", fy.output_root(REHEARSAL))

We start from `yolov8s`, the small pretrained (on the COCO dataset) model used in the course tutorial.

**Why the baseline writes out its optimizer.** With YOLO's default `optimizer=auto`, the training log says it is
*"ignoring lr0"*, so Philip's learning-rate runs would all train the same model. For a dataset our size, `auto` picks
AdamW with `lr0=0.001`, `momentum=0.9` and `warmup_bias_lr=0`, so `BASELINE` sets those explicitly. Behaviour stays the
same and `lr0` starts working.

**Other settings** are YOLO defaults (`imgsz=640`, `batch=16`). Epochs are 50 rather than 100, with `patience=20`, to fit
13+ runs on free Colab GPUs before the deadline. See the [YOLO train docs](https://docs.ultralytics.com/modes/train/#train-settings)
for what each hyperparameter does.

In [ ]:
settings = fy.run_settings(rehearsal=REHEARSAL)
model = YOLO(settings.pop("model"))  # pretrained model
print(settings)

The detailed architecture of the model can be seen from:

In [ ]:
model

Training. A finished baseline is not retrained if this cell is run again.

In [ ]:
baseline_weights = fy.weights_path("baseline", REHEARSAL)
if not baseline_weights.exists():
    model.train(data=str(DATA), project=str(fy.runs_dir(REHEARSAL)), name="baseline", exist_ok=True, **settings)
    baseline_weights = fy.record_run("baseline", "Daksh", model, rehearsal=REHEARSAL)

## Evaluation Metrics

[This guide](https://docs.ultralytics.com/guides/yolo-performance-metrics/) explains YOLO's performance metrics.

**Choosing the right metrics.** Our use case is finding rotten fruit before it spreads, so a missed rotten fruit costs more
than a false alarm. **Recall on the rotten classes** matters most, alongside mAP for overall quality.

- mAP: a broad assessment of model performance.
- Precision: how many flagged fruit were really that class. Low precision means staff check good fruit for nothing.
- Recall: how many real fruit were found. Low rotten recall means rot stays in the bin.
- F1 score: the balance between precision and recall.

In [ ]:
model = YOLO(str(baseline_weights))
metrics = model.val(data=str(DATA), imgsz=fy.trained_imgsz("baseline", REHEARSAL),
                    project=str(fy.runs_dir(REHEARSAL)), name="baseline_val", exist_ok=True)
print("mAP50-95:", metrics.box.map)
print("mAP50:   ", metrics.box.map50)
print("mAP75:   ", metrics.box.map75)
print("per class mAP50-95:", dict(zip(fy.CLASS_NAMES, metrics.box.maps.round(3).tolist())))
print("precision:", metrics.box.mp)
print("recall:   ", metrics.box.mr)

The rubric also asks for accuracy and F1, and our use case needs rotten-class results, so `evaluate` adds these:

- `accuracy`: correct boxes / (correct + wrong class + missed + false alarms), from the confusion matrix at confidence 0.25, the threshold a deployed system would use. Detection has no true negatives, so this is the closest equivalent. mAP is measured separately at YOLO's default low confidence, because it sweeps every threshold.
- `rotten_recall`: average recall of the three rotten classes.
- `rotten_flag_recall`: share of rotten fruit flagged as rotten of *any* type. Calling a rotten apple a rotten orange still gets it removed from the bin.

In [ ]:
baseline = fy.evaluate("baseline", baseline_weights, data=DATA, rehearsal=REHEARSAL)
display(pd.DataFrame([{k: baseline[k] for k in fy.METRIC_COLUMNS}], index=["baseline"]).round(3))
display(pd.DataFrame(baseline["per_class"]).T.round(3))
print({k: baseline[k] for k in ("rotten_called_fresh", "rotten_missed", "false_alarms")})

Showing the confusion matrix and training curves that YOLO saved.

In [ ]:
figures = fy.output_root(REHEARSAL) / "figures"
display(Image.open(figures / "baseline_val" / "confusion_matrix_normalized.png").resize((720, 540)))
display(Image.open(figures / "baseline" / "results.png").resize((1000, 500)))

Is the model underfitting or overfitting? The train and validation losses give the evidence.

In [ ]:
fy.fit_report("baseline", rehearsal=REHEARSAL)

Paste-ready numbers for the report, so the write-up uses exactly what was measured.

In [ ]:
fit = fy.fit_report("baseline", rehearsal=REHEARSAL)
print(pd.DataFrame([{k: baseline[k] for k in fy.METRIC_COLUMNS}], index=["baseline"]).round(3).to_markdown())
print()
print(pd.DataFrame(baseline["per_class"]).T.round(3).to_markdown())
print(f"\nFit: {fit['reading']} (lowest val loss at epoch {fit['best_epoch']} of {fit['epochs_run']}).")
print(f"Rotten fruit called fresh: {baseline['rotten_called_fresh']:.0f}. "
      f"Rotten fruit missed: {baseline['rotten_missed']:.0f}. False alarms: {baseline['false_alarms']:.0f}.")

### My baseline results

*TODO after the real run: report the metrics above, say whether the model is underfit or overfit, and give concrete fixes.*

## Correct and incorrect examples

Green boxes are the true labels, orange boxes are predictions.

In [ ]:
scored = fy.score_images(baseline_weights, root=ROOT, split="val")
wrong = sum(bool(s["problems"]) for s in scored)
print(f"{len(scored) - wrong} images fully correct, {wrong} with at least one mistake")
fy.show_examples(scored, correct=True, n=3, save_to=figures / "baseline" / "examples_correct.png"); plt.show()
fy.show_examples(scored, correct=False, n=3, save_to=figures / "baseline" / "examples_incorrect.png"); plt.show()

## Prediction

Running the saved model on a real photo that it never trained on, then showing the stored image, as the tutorial does.

In [ ]:
photos = kagglehub.dataset_download("mbkinaci/fruit-images-for-object-detection")
photo = sorted(glob.glob(f"{photos}/test_zip/test/mixed_*.jpg"))[0]

model = YOLO(str(baseline_weights))  # load the saved model
results = model(photo, save=True, project=str(fy.runs_dir(REHEARSAL)), name="baseline_predict", exist_ok=True)
Image.open(os.path.join(results[0].save_dir, os.path.basename(photo)))

# 6. Batch size tuning

Baseline `batch=16`, tuned at **8, 32 and 64**.

**What to expect.** YOLO accumulates gradients up to a nominal batch of 64 (`nbs=64`), so the weights update about every
64 images whichever batch we choose: 8 x 8, 16 x 4, 32 x 2, 64 x 1. Batch size then mainly changes BatchNorm statistics and
speed rather than the size of each update, so the differences may be small. The table records the effective batch for each
run, so this can be checked rather than assumed.

**Is a difference real?** The baseline fixes `seed=0`, so every run is repeatable, but a different seed would still
shuffle and augment the data differently. I train the baseline once more with `seed=1` (`baseline_seed1`) and use the
gap between the two as the run-to-run noise. A batch size only counts as better or worse if its gap to the baseline is
larger than that. One repeat is a rough estimate, so gaps close to it are treated as a tie. Other members can use the
same number through `fy.seed_noise`.

In [ ]:
OWNER = "Daksh"
PARAM, VALUES = fy.TUNING_PLAN[OWNER]
RUNS = [f"{PARAM}_{v}" for v in VALUES]

for name, value in zip(RUNS, VALUES):
    weights = fy.train_run(name, OWNER, overrides={PARAM: value}, data=DATA, rehearsal=REHEARSAL)
    fy.evaluate(name, weights, data=DATA, rehearsal=REHEARSAL)

# Run-to-run noise: the baseline again, changing only the random seed.
noise_weights = fy.train_run("baseline_seed1", OWNER, overrides={"seed": 1}, data=DATA, rehearsal=REHEARSAL)
fy.evaluate("baseline_seed1", noise_weights, data=DATA, rehearsal=REHEARSAL)

In [ ]:
table = fy.compare_runs(names=["baseline", "baseline_seed1"] + RUNS, rehearsal=REHEARSAL)
display(table)
noise = fy.seed_noise(table)
print(f"Seed noise on mAP@0.5:0.95: {noise:.4f}\n")
for name in RUNS:
    gap = table.loc[name, "mAP50_95_vs_baseline"]
    verdict = "larger than seed noise" if abs(gap) > noise else "within seed noise, so a tie"
    print(f"{name:<10} {gap:+.4f}  {verdict}")

title = "Batch size: mAP@0.5:0.95 on validation" + (" (REHEARSAL, not results)" if REHEARSAL else "")
fy.plot_comparison(table, title, noise=noise, save_to=figures / "tuning_batch.png")
plt.show()
for name in RUNS:
    print(name, fy.fit_report(name, rehearsal=REHEARSAL))

### My results

*TODO after the real run: which batch size won, by how much, and whether the gap is larger than run-to-run noise.*

# 7. Figures for the slides and GitHub

Every run's plots and confusion matrices are in one folder per run, so all slides use the same figures. On real runs this
is the shared Drive folder `IAT360-Group 18/results/figures`.

In [ ]:
for folder in sorted(p for p in figures.iterdir() if p.is_dir()):
    print(f"{folder.name:<20}", len(list(folder.glob("*.png"))), "plots")

Which runs from the whole group are trained, evaluated and have their plots in the shared folder. Anything at 0 is
still missing from the slides.

In [ ]:
fy.run_status(rehearsal=REHEARSAL)

The rubric needs results visible on GitHub. After a real run, this makes one zip in the repo's layout: unzip it at the top of the repo and commit.

In [ ]:
if REAL:
    fy.package_results(["baseline", "baseline_seed1"] + RUNS)
else:
    print("Rehearsal: nothing to publish.")

# 8. Notes for my report section

See [Daksh_Shortcomings_and_Challenges.md](https://github.com/laoneya31/IAT360-Group-18/blob/main/report/Daksh_Shortcomings_and_Challenges.md).
Update it with real numbers after Sections 5 and 6 have run on the group dataset.